In [ ]:
# ---------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------
import os
import time
from math import ceil
from Bio import SeqIO
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import pickle
import gc
from math import ceil


# ---------------------------------------------------------------------
# 1. One-hot utilities  ─ base encodings + helper functions
# ---------------------------------------------------------------------
def average_tensors(*tensors):
    """Average an arbitrary number of tensors (used for ambiguous bases)."""
    return sum(tensors) / len(tensors)


# Canonical bases
bases_encoding = {
    "A": torch.tensor([1., 0., 0., 0.]),
    "T": torch.tensor([0., 1., 0., 0.]),
    "C": torch.tensor([0., 0., 1., 0.]),
    "G": torch.tensor([0., 0., 0., 1.]),
    "N": torch.tensor([0., 0., 0., 0.])      # unknown / padding
}

# Ambiguous IUPAC bases encoded by averaging canonical vectors
ambiguous_bases = {
    "R": average_tensors(bases_encoding["A"], bases_encoding["G"]),
    "Y": average_tensors(bases_encoding["C"], bases_encoding["T"]),
    "K": average_tensors(bases_encoding["G"], bases_encoding["T"]),
    "S": average_tensors(bases_encoding["G"], bases_encoding["C"]),
    "W": average_tensors(bases_encoding["A"], bases_encoding["T"]),
    "M": average_tensors(bases_encoding["A"], bases_encoding["C"]),
    "B": average_tensors(bases_encoding["C"], bases_encoding["G"], bases_encoding["T"]),
    "D": average_tensors(bases_encoding["A"], bases_encoding["G"], bases_encoding["T"]),
    "H": average_tensors(bases_encoding["A"], bases_encoding["C"], bases_encoding["T"]),
    "V": average_tensors(bases_encoding["A"], bases_encoding["C"], bases_encoding["G"]),
}
bases_encoding.update(ambiguous_bases)


def one_hot_encode_custom(Xd):
    """
    Encode an array of windows (list[str]) → tensor [N, win_len, 4].
    """
    encoded = []
    for window in Xd:
        encoded_window = [
            bases_encoding.get(base, torch.tensor([0., 0., 0., 0.]))
            for base in window
        ]
        encoded.append(torch.stack(encoded_window))
    return torch.stack(encoded)

# ---------------------------------------------------------------------
# 2. Window creation for inference
# ---------------------------------------------------------------------


def create_input(seq, flanking_length, window_context):
    """
    Produce model-ready tensor [num_win, 4, win_len] from a single gene seq.
    """
    input_batch = []
    Y_length = len(seq)

    # Pad with N to give each window full context
    seq = 'N' * flanking_length + seq + 'N' * flanking_length
    X0 = np.array(list(seq), dtype=object)
    X0 = np.pad(X0, [0, flanking_length], mode='constant', constant_values='N')

    num_points = ceil(Y_length / flanking_length)
    Xd = np.full((num_points, flanking_length +
                 window_context), 'N', dtype=object)
    for i in range(num_points):
        Xd[i] = X0[flanking_length *
                   i: window_context + flanking_length * (i + 1)]

    X = one_hot_encode_custom(Xd)           # [num_win, win_len, 4]
    input_batch = torch.cat([X], dim=0)     # concat if multiple batches
    input_batch = input_batch.permute(0, 2, 1)  # → [num_win, 4, win_len]
    return input_batch

# ---------------------------------------------------------------------
# 3. Model definitions  (identical architecture to training)
# ---------------------------------------------------------------------


class SelfAttention(nn.Module):
    def __init__(self, hidden_unit: int):
        super().__init__()
        self.hidden_unit = hidden_unit
        self.query = nn.Linear(hidden_unit, hidden_unit)
        self.key = nn.Linear(hidden_unit, hidden_unit)
        self.value = nn.Linear(hidden_unit, hidden_unit)

    def forward(self, x: torch.Tensor):
        Q, K, V = self.query(x), self.key(x), self.value(x)
        attention_weights = F.softmax(
            Q @ K.transpose(-2, -1) / (self.hidden_unit ** 0.5), dim=-1
        )
        return attention_weights @ V


class ResidualCNNBlock(nn.Module):
    """Two conv-BN-ReLU layers + residual shortcut."""

    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(channels),
            nn.ReLU(),
            nn.Conv1d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(channels)
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(x + self.block(x))


class CNN_LSTM_Attention(nn.Module):
    """
    CNN → residual CNN → Bi-LSTM → self-attention → classifier
    """

    def __init__(self, input_shape, hidden_unit, lstm_hidden_unit, num_layers,
                 crop_size, num_classes, window_size=9, chunk_size=64):
        super().__init__()
        self.crop_size = crop_size
        # --- Conv front-end ---
        self.conv_block_1 = nn.Sequential(
            nn.Conv1d(input_shape, hidden_unit, kernel_size=3, padding=1),
            nn.BatchNorm1d(hidden_unit),
            nn.ReLU(),
            nn.Conv1d(hidden_unit, hidden_unit, kernel_size=3, padding=1),
            nn.BatchNorm1d(hidden_unit),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=3, stride=1, padding=1)
        )
        self.conv_block_2 = nn.Sequential(
            nn.Conv1d(hidden_unit, hidden_unit, kernel_size=3, padding=1),
            nn.BatchNorm1d(hidden_unit),
            nn.ReLU(),
            nn.Conv1d(hidden_unit, hidden_unit, kernel_size=3, padding=1),
            nn.BatchNorm1d(hidden_unit),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=3, stride=1, padding=1)
        )
        self.cnn_backbone = nn.Sequential(
            ResidualCNNBlock(hidden_unit),
            ResidualCNNBlock(hidden_unit),
            ResidualCNNBlock(hidden_unit),
        )
        # --- LSTM + attention ---
        self.lstm = nn.LSTM(
            hidden_unit, lstm_hidden_unit, num_layers,
            batch_first=True, bidirectional=True
        )
        self.attention = SelfAttention(hidden_unit=2 * lstm_hidden_unit)
        self.norm1 = nn.LayerNorm(2 * lstm_hidden_unit)
        self.classifier = nn.Linear(2 * lstm_hidden_unit, num_classes)

    def forward(self, x):
        x = self.conv_block_1(x)
        x = self.conv_block_2(x)
        x = self.cnn_backbone(x)
        x = x.transpose(1, 2)      # [B, L, C]
        x, _ = self.lstm(x)
        x = self.attention(x)
        x = self.norm1(x)
        x = self.classifier(x)     # [B, L, 3]
        x = x.transpose(1, 2)      # [B, 3, L]

        # Center-crop to remove padded context
        left_crop = self.crop_size // 2
        right_crop = self.crop_size - left_crop
        if right_crop > 0:
            x = x[:, :, left_crop:-right_crop]
        else:
            x = x[:, :, left_crop:]
        return x

# ---------------------------------------------------------------------
# 4. Ensemble inference for a FASTA file
# ---------------------------------------------------------------------


def gather_predictions_once(models, device, window_context, fastafile, batch_size):
    """
    For each gene record in FASTA, run an ensemble of models and
    return dict{geneid: {true labels, predicted labels, donor/acceptor probs}}.
    """
    pred_data = {}
    records = list(SeqIO.parse(fastafile, "fasta"))

    for record in tqdm(records, desc="Loading:"):
        seq = record.seq.upper()
        seq_info = str(record.name)

        # Unpack FASTA header (matches encoding during dataset creation)
        geneid = seq_info.split("_")[0]
        intron_start_index_str = seq_info.split("_")[-2]
        intron_end_index_str = seq_info.split("_")[-1]
        intron_start_index = intron_start_index_str.split(",")
        intron_end_index = intron_end_index_str.split(",")
        gene_end, gene_start = seq_info.split("_")[-3], seq_info.split("_")[-4]
        strand, chromosome = seq_info.split("_")[-5], seq_info.split("_")[-6]
        geneid = seq_info.split(
            f"_{chromosome}_{strand}_{gene_start}_{gene_end}_{intron_start_index_str}_{intron_end_index_str}"
        )[0]

        # Skip genes with no introns
        if not intron_start_index[0] or not intron_end_index[0]:
            continue

        intron_start_list = [int(x) for x in intron_start_index]
        intron_end_list = [int(x) for x in intron_end_index]

        flanking_length = window_context // 2
        X_torch = create_input(seq, flanking_length, window_context).to(device)

        # ---------- Ensemble prediction ----------
        with torch.inference_mode():
            probs_list = []
            for model in models:
                sub_preds = []
                for start in range(0, X_torch.size(0), batch_size):
                    chunk = model(X_torch[start: start + batch_size])
                    chunk = chunk.transpose(1, 2)          # [B, L, 3]
                    sub_preds.append(F.softmax(chunk, dim=-1))
                probs_list.append(torch.cat(sub_preds, dim=0))
            probs = torch.mean(torch.stack(probs_list), dim=0)  # avg ensemble

        # ---------- Post-processing ----------
        y = probs.reshape(-1, 3)
        predicted_labels = torch.argmax(y, dim=1).cpu().numpy()
        acceptor_prob = y[:, 1].cpu().numpy()
        donor_prob = y[:, 2].cpu().numpy()

        # Build ground-truth label array (0=no-splice, 1=acc, 2=don)
        padded_true_label = np.zeros(len(predicted_labels), dtype=int)
        for pos in intron_start_list:
            padded_true_label[int(pos)] = 2                # donor
        for pos in intron_end_list:
            padded_true_label[int(pos)] = 1                # acceptor

        pred_data[geneid] = {
            'True_label': padded_true_label,
            'Predict_label': predicted_labels,
            'Donor_prob': donor_prob,
            'Acceptor_prob': acceptor_prob
        }

    return pred_data


# ---------------------------------------------------------------------
# 5. Batch prediction over validation sets
# ---------------------------------------------------------------------
model_list = ['Arabidopsis_thaliana']
species_list = ['Arabidopsis_thaliana']
batch_size = 32
filetypelist = ['no_AS']
base_dir = '/results/Step4_Top_K_accuracy_calculation_code'

for modelname in model_list:
    for species in species_list:
        for filetype in filetypelist:

            validation_dir = f'/data/Step2_generate_training_data/{species}'
            device = 'cuda'
            window_context = 600
            epoch_type_list = [1, 2, 3, 4, 5]     # ensemble of 5 seeds

            # ---------- Load ensemble models ----------
            models = []
            for epoch in epoch_type_list:
                model = CNN_LSTM_Attention(
                    input_shape=4,
                    hidden_unit=60,
                    lstm_hidden_unit=60,
                    num_layers=3,
                    crop_size=window_context,
                    num_classes=3
                ).to(device)
                weight_path = (f"/data/Pretrained_models/"
                               f"{modelname}_{window_context}_{epoch}.pt")
                model.load_state_dict(torch.load(
                    weight_path, weights_only=True))
                model.eval()
                models.append(model)

            # ---------- Predict for FASTA ----------
            fastafile = f'{validation_dir}/{species}_{filetype}_test_gene_sequences.fasta'
            pred_data = gather_predictions_once(models, device, window_context,
                                                fastafile, batch_size)

            # ---------- Save predictions ----------
            os.makedirs(f"{base_dir}/pkl_file", exist_ok=True)
            out_path = (f"{base_dir}/pkl_file/{modelname}_model_pred_on_"
                        f"{species}_{filetype}_dataset.pkl")
            with open(out_path, "wb") as f:
                pickle.dump(pred_data, f)
            print(f"Saved {os.path.basename(out_path)}")

            # ---------- Clean up ----------
            del models                  # drop references to models
            torch.cuda.empty_cache()    # free GPU memory
            gc.collect()                # run garbage collection

In [ ]:
def splice_site_accuracy(
        pkl_file: str,
        *,
        mode: str = "topk",      # "topk"  | "threshold"
        # int or float∈(0,1]; ignored when mode="threshold"
        top_number=1,
        topk: bool = False,     # reproduce legacy behaviour (k = #true sites)
        threshold: float = 0.5  # score threshold when mode=="threshold"
):
    """
    Compute splice-site accuracy from a *.pkl prediction file.
    Supports two evaluation modes:
      • mode="threshold": take *all* predictions ≥ threshold.
      • mode="topk":      take top-k predictions by probability, where
            topk=True          → k = #true sites (legacy)
            top_number=float   → proportion of actual sites   (0 < f ≤ 1)
            top_number=int ≥1  → absolute number per class.
    The function prints and returns the overall accuracy.
    """

    # ------------------------------------------------------------------
    # Load predictions dict  {geneid: {...}}
    # ------------------------------------------------------------------
    with open(pkl_file, "rb") as f:
        pred_data = pickle.load(f)

    total_correct = 0
    total_positions = 0   # denominator depends on mode (see below)

    # ------------------------------------------------------------------
    # 1) HARD-THRESHOLD evaluation (identical to top_k_hard_accuracy)
    # ------------------------------------------------------------------
    if mode == "threshold":
        for geneid, gene_data in pred_data.items():
            true_label = gene_data['True_label']
            pred_label = gene_data['Predict_label']
            pred_donor_rate = gene_data['Donor_prob'].tolist()
            pred_acceptor_rate = gene_data['Acceptor_prob'].tolist()

            # Counts & ground-truth site lists
            donor_actual_number = np.sum(true_label == 2)
            acceptor_actual_number = np.sum(true_label == 1)
            total_splice_sites = donor_actual_number + acceptor_actual_number
            true_donor_site = np.where(true_label == 2)[0].tolist()
            true_acceptor_site = np.where(true_label == 1)[0].tolist()

            # All predicted donor / acceptor positions
            pred_donor_site = np.where(pred_label == 2)[0].tolist()
            pred_acceptor_site = np.where(pred_label == 1)[0].tolist()

            # Filter by probability ≥ threshold, then sort by score ↓
            coordinates_donor = sorted(
                [(pos, pred_donor_rate[pos], 1)
                 for pos in pred_donor_site if pred_donor_rate[pos] >= threshold],
                key=lambda x: x[1], reverse=True)

            coordinates_acceptor = sorted(
                [(pos, pred_acceptor_rate[pos], 0)
                 for pos in pred_acceptor_site if pred_acceptor_rate[pos] >= threshold],
                key=lambda x: x[1], reverse=True)

            # Keep top k, where k = #actual sites
            top_n_donor = coordinates_donor[:donor_actual_number]
            top_n_acceptor = coordinates_acceptor[:acceptor_actual_number]

            # Convert to pure position lists
            top_n_donor_list = sorted([item[0] for item in top_n_donor])
            top_n_acceptor_list = sorted([item[0] for item in top_n_acceptor])
            total_n_donor_list = sorted([item[0]
                                        for item in coordinates_donor])
            total_n_acceptor_list = sorted(
                [item[0] for item in coordinates_acceptor])

            # True positives / false positives
            correct_donor = len(set(top_n_donor_list) & set(true_donor_site))
            correct_acceptor = len(
                set(top_n_acceptor_list) & set(true_acceptor_site))
            false_donor_count = len(total_n_donor_list) - correct_donor
            false_acceptor_count = len(
                total_n_acceptor_list) - correct_acceptor

            # Adjust for double-counting
            correct_site_num = (correct_donor + correct_acceptor
                                - false_donor_count - false_acceptor_count)

            total_positions += total_splice_sites
            total_correct += correct_site_num

        acc = total_correct / total_positions
        print(acc)
        return acc

    # ------------------------------------------------------------------
    # 2) TOP-K evaluation  (legacy + proportional + absolute)
    # ------------------------------------------------------------------
    for geneid, gene_data in pred_data.items():
        true_label = gene_data['True_label']
        pred_label = gene_data['Predict_label']
        pred_donor_rate = gene_data['Donor_prob'].tolist()
        pred_acceptor_rate = gene_data['Acceptor_prob'].tolist()

        # Ground-truth counts & positions
        donor_actual_number = np.sum(true_label == 2)
        acceptor_actual_number = np.sum(true_label == 1)
        total_splice_sites = donor_actual_number + acceptor_actual_number
        true_donor_site = np.where(true_label == 2)[0].tolist()
        true_acceptor_site = np.where(true_label == 1)[0].tolist()

        # All predicted site coordinates sorted by probability ↓
        pred_donor_site = np.where(pred_label == 2)[0].tolist()
        pred_acceptor_site = np.where(pred_label == 1)[0].tolist()
        coordinates_donor = sorted(
            [(pos, pred_donor_rate[pos], 1) for pos in pred_donor_site],
            key=lambda x: x[1], reverse=True)
        coordinates_acceptor = sorted(
            [(pos, pred_acceptor_rate[pos], 0) for pos in pred_acceptor_site],
            key=lambda x: x[1], reverse=True)

        # ---------- 2a) Legacy: k = #true sites -------------------------
        if topk:
            top_n_donor = coordinates_donor[:donor_actual_number]
            top_n_acceptor = coordinates_acceptor[:acceptor_actual_number]
            top_n_donor_list = sorted([item[0] for item in top_n_donor])
            top_n_acceptor_list = sorted([item[0] for item in top_n_acceptor])

            correct_donor = len(set(top_n_donor_list) & set(true_donor_site))
            correct_acceptor = len(
                set(top_n_acceptor_list) & set(true_acceptor_site))
            total_correct += (correct_donor + correct_acceptor)
            total_positions += total_splice_sites
            continue

        # ---------- 2b) Proportional k (0 < f ≤ 1) ----------------------
        if isinstance(top_number, float) and 0 < top_number <= 1:
            k_donor = ceil(donor_actual_number * top_number)
            k_acceptor = ceil(acceptor_actual_number * top_number)

            top_n_donor = coordinates_donor[:k_donor]
            top_n_acceptor = coordinates_acceptor[:k_acceptor]
            top_n_donor_list = sorted([item[0] for item in top_n_donor])
            top_n_acceptor_list = sorted([item[0] for item in top_n_acceptor])

            correct_donor = len(set(top_n_donor_list) & set(true_donor_site))
            correct_acceptor = len(
                set(top_n_acceptor_list) & set(true_acceptor_site))
            total_correct += (correct_donor + correct_acceptor)
            total_positions += (k_donor + k_acceptor)
            continue

        # ---------- 2c) Absolute k (integer ≥ 1) ------------------------
        if isinstance(top_number, int) and top_number >= 1:
            top_n_donor = coordinates_donor[:top_number]
            top_n_acceptor = coordinates_acceptor[:top_number]
            top_n_donor_list = sorted([item[0] for item in top_n_donor])
            top_n_acceptor_list = sorted([item[0] for item in top_n_acceptor])

            correct_donor = len(set(top_n_donor_list) & set(true_donor_site))
            correct_acceptor = len(
                set(top_n_acceptor_list) & set(true_acceptor_site))
            total_correct += (correct_donor + correct_acceptor)

            # Denominator depends on available sites vs requested k
            if top_number <= min(donor_actual_number, acceptor_actual_number):
                total_positions += 2 * top_number
            elif top_number <= max(donor_actual_number, acceptor_actual_number):
                total_positions += min(donor_actual_number,
                                       acceptor_actual_number) + top_number
            else:
                total_positions += total_splice_sites
            continue

        # If none of the branches matched:
        raise ValueError("top_number must be an int ≥ 1, a float in (0,1], "
                         "or use mode='threshold'.")

    acc = f'{total_correct / total_positions}'
    print(acc)
    return acc


species_list = ['Arabidopsis_thaliana']

for species in species_list:
    pkl_file_dir = '/results/Step4_Top_K_accuracy_calculation_code/pkl_file'

    as_pkl = f"{pkl_file_dir}/{species}_model_pred_on_{species}_no_AS_dataset.pkl"
    acc = splice_site_accuracy(as_pkl, mode="topk", top_number=1)
    acc = splice_site_accuracy(as_pkl, mode="topk", top_number=5)
    acc = splice_site_accuracy(as_pkl, mode="topk", top_number=0.5)
    acc = splice_site_accuracy(as_pkl, mode="topk", topk=True)